# ADIKARA 2026 Data Mining: Tweet Likes Prediction

Predicts `likes` for every row of `test.csv`. Evaluation metric: RMSLE.

**Approach**
1. Clean the data: the `likes` text suffixes, three `post_date` formats, category typos and brand-handle casing.
2. Engineer features from date, text statistics and account behaviour (unsupervised, train + test features only).
3. Encode text with TF-IDF and two public pretrained encoders (IndoBERTweet, multilingual E5), used for inference only.
4. Stack out-of-fold linear text models and out-of-fold account statistics as features, using nested cross-validation so no row sees its own label.
5. Train gradient-boosted models: LightGBM, a LightGBM hurdle model, XGBoost and CatBoost. The target is `log1p(likes)` with squared error, which is equivalent to optimising RMSLE.
6. Blend with non-negative weights summing to 1. The weights are validated with a second-level CV, and the blend writes `submission.csv`.

**How to run:** put `train.csv`, `test.csv` and `sample_submission.csv` in a `data/` folder next to this notebook or one level up, then run all cells. Runtime is about 30–60 min on CPU; a GPU (CUDA or Apple MPS) speeds up the embedding step. No external data is used, no API keys are needed, and no model is fine-tuned.

## Sources and dependencies

**Pretrained models** (public, ungated Hugging Face repositories; used only to compute text embeddings, without fine-tuning)

| Model | Repository | Licence | Reference |
|---|---|---|---|
| IndoBERTweet (base, uncased) | https://huggingface.co/indolem/indobertweet-base-uncased | Apache-2.0 | Koto, F., Lau, J. H., & Baldwin, T. (2021). *IndoBERTweet: A Pretrained Language Model for Indonesian Twitter with Effective Domain-Specific Vocabulary Initialization.* EMNLP 2021. https://aclanthology.org/2021.emnlp-main.833 |
| Multilingual E5 (base) | https://huggingface.co/intfloat/multilingual-e5-base | MIT | Wang, L. et al. (2024). *Multilingual E5 Text Embeddings: A Technical Report.* arXiv:2402.05672 |

**Libraries and dependencies** (versions used for the submitted run; the next code cell prints the versions of the running environment)

| Library | Version | Reference |
|---|---|---|
| Python | 3.13.2 | |
| pandas | 3.0.3 | |
| numpy | 2.5.1 | |
| scipy | 1.18.1 | |
| pyarrow | 24.0.0 | |
| scikit-learn | 1.9.1 | Pedregosa et al. (2011), JMLR 12 |
| LightGBM | 4.7.0 | Ke et al. (2017), *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*, NeurIPS 2017 |
| XGBoost | 3.4.1 | Chen & Guestrin (2016), *XGBoost: A Scalable Tree Boosting System*, KDD 2016 |
| CatBoost | 1.2.10 | Prokhorenkova et al. (2018), *CatBoost: unbiased boosting with categorical features*, NeurIPS 2018, arXiv:1706.09516 |
| PyTorch | 2.15.0 | |
| transformers | 5.22.2 | |
| emoji | 2.16.0 | used for the IndoBERTweet preprocessing (emoji to text) |

Install: `pip install pandas numpy scipy pyarrow scikit-learn lightgbm xgboost catboost torch transformers emoji`

In [1]:
import importlib.util, subprocess, sys
missing = [p for p, m in [("emoji", "emoji"), ("lightgbm", "lightgbm"), ("xgboost", "xgboost"),
                          ("catboost", "catboost"), ("transformers", "transformers")]
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import json, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy, sklearn, lightgbm as lgb, xgboost as xgb, catboost, torch, transformers, emoji
from scipy import sparse
from scipy.optimize import minimize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import KFold, StratifiedKFold

warnings.filterwarnings("ignore")
SEED = 42
N_FOLDS = 5
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "data" / "train.csv").exists())
DATA, CACHE, OUT = ROOT / "data", ROOT / "cache" / "final_nb", ROOT / "submission"
CACHE.mkdir(parents=True, exist_ok=True); OUT.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

for name, mod in [("python", sys), ("pandas", pd), ("numpy", np), ("scipy", scipy), ("scikit-learn", sklearn),
                  ("lightgbm", lgb), ("xgboost", xgb), ("catboost", catboost), ("torch", torch),
                  ("transformers", transformers), ("emoji", emoji)]:
    print(f"{name:13s}", sys.version.split()[0] if mod is sys else mod.__version__)
print("device:", DEVICE, "| data:", DATA)
T0 = time.time()

python        3.12.4
pandas        2.3.3
numpy         1.26.4
scipy         1.15.2
scikit-learn  1.4.2
lightgbm      4.7.0
xgboost       3.4.1
catboost      1.2.10
torch         2.14.0+cpu
transformers  5.17.0
emoji         2.16.0
device: cpu | data: c:\Users\nayaj\Downloads\adikara-2026-tweet-likes\data


## 1. Load and clean the data

Data issues found during the audit:
- **`likes`:** 396 values carry a text suffix (`"140 pts"`, `"128 score"`, `"0 likes"`), and the suffix is stripped. 25 values are negative (`-1`, `-99`); they are placeholders for missing values, not like counts, so those rows are dropped from training (0.1%).
- **`post_date`:** values come in three formats (ISO with offset, `DD/MM/YYYY hh:mm:ss`, Unix epoch seconds), and all are parsed to UTC. Epoch-0 placeholders (1970) become missing.
- **Categories:** typos are mapped to the correct value (`foto`, `vidio`, `carousell`, `Andorid`, `Web-App`), and missing values become a `missing` category.
- **Usernames:** brand handles differ only by case (`@AuraMobileID`, `@auramobileid`), so they are lowercased. Hashed usernames are left unchanged.

In [2]:
def parse_post_date(s):
    s = s.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    iso = s.str.match(r"^\d{4}-\d{2}-\d{2}")
    dmy = s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}")
    epo = s.str.match(r"^\d{9,11}$")
    out[iso] = pd.to_datetime(s[iso], utc=True).dt.tz_localize(None).astype("datetime64[ns]")
    out[dmy] = pd.to_datetime(s[dmy], format="%d/%m/%Y %H:%M:%S").astype("datetime64[ns]")
    out[epo] = pd.to_datetime(s[epo].astype("int64"), unit="s").astype("datetime64[ns]")
    return out


def date_format(s):
    s = s.astype(str).str.strip()
    return np.select([s.str.match(r"^\d{4}-"), s.str.match(r"^\d{1,2}/"), s.str.match(r"^\d{9,11}$")],
                     ["iso", "dmy", "epoch"], "other")


def clean_likes(s):
    v = pd.to_numeric(s.astype(str).str.strip().str.replace(r"\s*(pts|score|likes)$", "", regex=True), errors="coerce")
    return v.where(v >= 0)


MEDIA_FIX = {"foto": "photo", "vidio": "video", "carousell": "carousel"}
DEVICE_FIX = {"Andorid": "Android", "Web-App": "Web App"}


def clean_cat(s, fix):
    s = s.where(s.isna(), s.astype(str).str.strip()).replace(fix)
    return s.fillna("missing")


def clean_username(s):
    s = s.astype(str).str.strip()
    return s.where(~s.str.startswith("@"), s.str.lower())


train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")

train["likes"] = clean_likes(train["likes"])
print("dropped rows with negative / unparseable likes:", train["likes"].isna().sum())
train = train[train["likes"].notna()].reset_index(drop=True)
for d in (train, test):
    d["date_fmt"] = date_format(d["post_date"])
    d["dt"] = parse_post_date(d["post_date"])
    d.loc[d["dt"] < "2000-01-01", "dt"] = pd.NaT
    d["text"] = d["text"].fillna("").astype(str)
    d["username"] = clean_username(d["username"])
    d["media_type"] = clean_cat(d["media_type"], MEDIA_FIX)
    d["client_device"] = clean_cat(d["client_device"], DEVICE_FIX)

y = np.log1p(train["likes"].to_numpy())
print("train", train.shape, "test", test.shape)
print(f"log1p(likes): mean {y.mean():.3f}, std {y.std():.3f}, share of zero likes {(y == 0).mean():.3f}")
print("constant-prediction RMSLE baseline:", round(y.std(), 5))

dropped rows with negative / unparseable likes: 25
train (26163, 9) test (5969, 8)
log1p(likes): mean 3.917, std 2.336, share of zero likes 0.257
constant-prediction RMSLE baseline: 2.33586


## 2. Cross-validation folds

The audit found a random split. Test makes up about 18–19% of every month, and 31% of test rows come from accounts not seen in train; random 5-fold CV reproduces this at 32.6%. The folds are therefore a `StratifiedKFold` on bins of `log1p(likes)` (a zero bin plus deciles of the non-zero values), and they are fixed for every model.

In [3]:
bins = np.zeros(len(y), dtype=int)
nz = y > 0
bins[nz] = 1 + pd.qcut(y[nz], 10, labels=False, duplicates="drop")
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(y, bins))


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


unseen = [1 - train.username.iloc[v].isin(set(train.username.iloc[t])).mean() for t, v in folds]
print("validation rows with an unseen account:", np.round(unseen, 3),
      "| test rows with an unseen account:", round(1 - test.username.isin(set(train.username)).mean(), 3))

validation rows with an unseen account: [0.324 0.331 0.32  0.325 0.332] | test rows with an unseen account: 0.312


## 3. Unsupervised features (computed on train + test features, never on `likes`)

- **Date:** year, month, day, hour, minute, weekday, weekend flag, days since the first post, hour of week, seconds of day, and a round-time flag for scheduled posts.
- **Text statistics:** length, word count, hashtags, mentions, URLs, emojis, uppercase and digit ratios, punctuation, `?` and `!`, elongated words, RT and reply flags, base/fess tags, short-text flag.
- **Account:** post count, activity span and frequency, rank of the post within the account, gaps to the account's previous and next posts, the account's media and device mix, text duplication, posts on the same day.

In [4]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+|\b[\w-]+\.(?:com|id|co|net|org|ly|me)(?:/\S*)?)", re.I)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
EMOJI_RE = re.compile("[\U0001F000-\U0001FAFF☀-➿⬀-⯿️]")
ELONG_RE = re.compile(r"(\w)\1{2,}")
TAG_RE = re.compile(r"^\s*[/\[(]\s*[\w-]+\s*[/\])]")
CATS = ["media_type", "client_device", "date_fmt", "acct_type"]


def text_stats(t):
    t = t.astype(object)
    n = t.str.len()
    letters = t.str.count(r"[A-Za-z]").clip(lower=1)
    return pd.DataFrame({
        "len_chars": n, "len_words": t.str.split().str.len().fillna(0),
        "n_hashtags": t.str.count(HASHTAG_RE), "n_mentions": t.str.count(MENTION_RE),
        "n_urls": t.str.count(URL_RE), "n_emojis": t.str.count(EMOJI_RE),
        "upper_ratio": t.str.count(r"[A-Z]") / letters, "digit_ratio": t.str.count(r"\d") / n.clip(lower=1),
        "n_punct": t.str.count(r"[^\w\s]"), "n_question": t.str.count(r"\?"), "n_exclam": t.str.count(r"!"),
        "n_elong": t.str.count(ELONG_RE), "is_rt": t.str.match(r"^\s*RT\b").astype(int),
        "starts_mention": t.str.match(r"^\s*@").astype(int), "has_base_tag": t.str.match(TAG_RE).astype(int),
        "is_short": (n < 20).astype(int), "has_ellipsis_link": t.str.contains("…").astype(int),
    })


def basic_features(tr, te):
    al = pd.concat([tr.drop(columns=["likes"]), te], ignore_index=True)
    dt = al["dt"]
    f = pd.DataFrame(index=al.index)
    f["media_type"], f["client_device"], f["date_fmt"] = al.media_type, al.client_device, al.date_fmt
    f["acct_type"] = np.where(al.username.str.startswith("@"), al.username, "hashed")
    f["is_brand"] = al.username.str.startswith("@").astype(int)
    f["year"], f["month"], f["day"] = dt.dt.year, dt.dt.month, dt.dt.day
    f["hour"], f["minute"], f["weekday"] = dt.dt.hour, dt.dt.minute, dt.dt.weekday
    f["is_weekend"] = (dt.dt.weekday >= 5).astype(float).where(dt.notna())
    f["days"] = (dt - dt.min()).dt.total_seconds() / 86400
    f["hour_of_week"] = dt.dt.weekday * 24 + dt.dt.hour
    f["sec_of_day"] = dt.dt.hour * 3600 + dt.dt.minute * 60 + dt.dt.second
    f["is_round_time"] = ((dt.dt.second == 0) & (dt.dt.minute % 5 == 0)).astype(float).where(dt.notna())
    f = f.join(text_stats(al.text))

    u, g = al.username, al.groupby("username")
    f["acct_n"] = u.map(u.value_counts())
    first, last = g["dt"].transform("min"), g["dt"].transform("max")
    span = (last - first).dt.total_seconds() / 86400
    f["acct_span_days"] = span
    f["acct_freq"] = f["acct_n"] / (span + 1)
    f["acct_rank"] = g["dt"].rank(method="first") / f["acct_n"]
    f["acct_days_since_first"] = (dt - first).dt.total_seconds() / 86400
    srt = al.sort_values(["username", "dt"])
    f["gap_prev_h"] = (srt.groupby("username")["dt"].diff().dt.total_seconds() / 3600).reindex(al.index)
    f["gap_next_h"] = (-srt.groupby("username")["dt"].diff(-1).dt.total_seconds() / 3600).reindex(al.index)
    for c, vals in [("media_type", ["text_only", "photo", "video", "carousel"]),
                    ("client_device", ["Android", "iOS", "Web App", "Lite App"])]:
        for v in vals:
            f[f"acct_share_{c}_{v}".replace(" ", "_")] = (al[c] == v).groupby(u).transform("mean")
    norm_text = al.text.str.lower().str.replace(URL_RE, "<url>", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    f["text_dup_global"] = norm_text.map(norm_text.value_counts())
    f["acct_text_dup_rate"] = 1 - norm_text.groupby(u).transform("nunique") / f["acct_n"]
    f["acct_mean_len"] = f.groupby(u)["len_chars"].transform("mean")
    f["acct_mean_hashtags"] = f.groupby(u)["n_hashtags"].transform("mean")
    day = dt.dt.floor("D")
    f["acct_posts_same_day"] = al.assign(_d=day).groupby(["username", "_d"])["id"].transform("count")
    f["global_posts_same_day"] = day.map(day.value_counts())
    for c in CATS:
        f[c] = f[c].astype("category")
    n = len(tr)
    return f.iloc[:n].reset_index(drop=True), f.iloc[n:].reset_index(drop=True)


Xtr, Xte = basic_features(train, test)
print("basic features:", Xtr.shape[1])

basic features: 54


## 4. Text representations

- **TF-IDF:** word 1–2-grams plus character 2–5-grams (`char_wb`). URLs become `<url>` and mentions become `<user>`. The vocabulary is fit on train + test text, which uses no labels.
- **Pretrained encoders:** the mean-pooled last hidden state (max length 128) of IndoBERTweet and multilingual E5. IndoBERTweet input follows its model card: lowercase, mentions as `@USER`, URLs as `HTTPURL`, emojis converted to text. E5 input uses the `query: ` prefix. Embeddings are cached in `cache/final_nb/`, so reruns skip this step.

In [5]:
def norm_for_tfidf(t):
    t = t.astype(object).str.lower()
    t = t.str.replace(URL_RE, " <url> ", regex=True).str.replace(MENTION_RE, " <user> ", regex=True)
    return t.str.replace(r"\s+", " ", regex=True).str.strip()


txt_all = pd.concat([norm_for_tfidf(train.text), norm_for_tfidf(test.text)], ignore_index=True)
w_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True,
                        token_pattern=r"(?u)<\w+>|#?\b\w+\b")
c_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=300_000, sublinear_tf=True)
M = sparse.hstack([w_vec.fit_transform(txt_all), c_vec.fit_transform(txt_all)]).tocsr()
Mtr, Mte = M[:len(train)], M[len(train):]
print("TF-IDF matrix:", M.shape)

TF-IDF matrix: (32132, 209673)


In [6]:
from transformers import AutoModel, AutoTokenizer

ENCODERS = {"indobertweet": "indolem/indobertweet-base-uncased", "e5": "intfloat/multilingual-e5-base"}


def prep_indobertweet(t):
    t = t.lower()
    t = re.sub(r"@\w+", "@USER", t)
    t = re.sub(r"(https?://\S+|www\.\S+|\b[\w-]+\.(?:com|id|co|net|org|ly|me)(?:/\S*)?)", "HTTPURL", t)
    return re.sub(r"\s+", " ", emoji.demojize(t)).strip()


def prep_e5(t):
    return "query: " + re.sub(r"\s+", " ", t).strip()


@torch.no_grad()
def encode(texts, repo, batch_size=64, max_len=128):
    tok = AutoTokenizer.from_pretrained(repo)
    model = AutoModel.from_pretrained(repo).to(DEVICE).eval()
    if DEVICE == "cuda":
        model = model.half()
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        enc = tok([texts[j] for j in idx], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
        h = model(**enc).last_hidden_state.float()
        m = enc["attention_mask"].unsqueeze(-1).float()
        out[idx] = ((h * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
    return out


EMB = {}
raw_text = list(train.text) + list(test.text)
for name, repo in ENCODERS.items():
    path = CACHE / f"emb_{name}.npy"
    if path.exists():
        E = np.load(path).astype(np.float32)
    else:
        t0 = time.time()
        prep = prep_indobertweet if name == "indobertweet" else prep_e5
        E = encode([prep(t) for t in raw_text], repo)
        np.save(path, E.astype(np.float16))
        print(f"{name}: encoded in {time.time() - t0:.0f}s")
    E = E.astype(np.float32)
    E = (E - E.mean(0)) / (E.std(0) + 1e-6)  # standardisation fit on train + test features only
    EMB[name] = (E[:len(train)], E[len(train):])
    print(name, repo, E.shape)

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  445MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (12 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

e5: encoded in 2448s
e5 intfloat/multilingual-e5-base (32132, 768)


## 5. Out-of-fold stacked features (nested, leak-free)

Each outer fold computes features from its own training part only:
- **Account target statistics:** each account's count, smoothed mean (m = 20), smoothed zero rate, mean, std, max, median and non-zero mean of `log1p(likes)`. Training rows get these from an inner 5-fold split, so a row never sees its own label; validation and test rows use the whole outer-training part.
- **Linear text models:** a ridge regression (`log1p(likes)`) and a logistic regression (zero likes) fit on TF-IDF, IndoBERTweet and E5. They follow the same nested scheme: inner out-of-fold predictions for training rows, and a fit on the outer-training part for validation and test rows.

For each fold model, the test features are computed from that fold's training part, the same way as the validation features.

In [7]:
def account_stats(keys_src, y_src, keys_dst, m=20, prefix="te_user"):
    gm = float(np.mean(y_src))
    d = pd.DataFrame({"k": keys_src.to_numpy(), "y": y_src, "z": (y_src == 0).astype(float)})
    g = d.groupby("k").agg(n=("y", "size"), mean=("y", "mean"), std=("y", "std"), mx=("y", "max"),
                           med=("y", "median"), zr=("z", "mean"),
                           nzmean=("y", lambda s: s[s > 0].mean() if (s > 0).any() else np.nan))
    kd = pd.Series(keys_dst.to_numpy())
    n = kd.map(g["n"]).fillna(0).to_numpy()
    return pd.DataFrame({
        f"{prefix}_n": n,
        f"{prefix}_mean_s": (kd.map(g["mean"]).fillna(0).to_numpy() * n + m * gm) / (n + m),
        f"{prefix}_zero_s": (kd.map(g["zr"]).fillna(0).to_numpy() * n + m * (y_src == 0).mean()) / (n + m),
        f"{prefix}_mean": kd.map(g["mean"]).to_numpy(), f"{prefix}_std": kd.map(g["std"]).to_numpy(),
        f"{prefix}_max": kd.map(g["mx"]).to_numpy(), f"{prefix}_median": kd.map(g["med"]).to_numpy(),
        f"{prefix}_nzmean": kd.map(g["nzmean"]).to_numpy(),
    })


def account_stats_inner(keys, y_, inner_folds=5, seed=SEED):
    fold = np.random.RandomState(seed).randint(0, inner_folds, len(y_))
    out = None
    for k in range(inner_folds):
        a, b = fold != k, fold == k
        part = account_stats(keys[a], y_[a], keys[b])
        if out is None:
            out = pd.DataFrame(index=range(len(y_)), columns=part.columns, dtype=float)
        out.loc[np.where(b)[0]] = part.to_numpy()
    return out.astype(float)


def linear_stack(M_trn, y_trn, M_dsts, prefix, alpha, C, inner_folds=5, seed=SEED):
    def fit_pred(Ma, ya, Mbs):
        r = Ridge(alpha=alpha).fit(Ma, ya)
        lr = LogisticRegression(C=C, max_iter=3000).fit(Ma, (ya == 0).astype(int))
        return [np.c_[r.predict(Mb), lr.predict_proba(Mb)[:, 1]] for Mb in Mbs]
    oof = np.zeros((M_trn.shape[0], 2))
    for a, b in KFold(inner_folds, shuffle=True, random_state=seed).split(y_trn):
        oof[b] = fit_pred(M_trn[a], y_trn[a], [M_trn[b]])[0]
    cols = [f"{prefix}_ridge", f"{prefix}_pzero"]
    return [pd.DataFrame(x, columns=cols) for x in [oof] + fit_pred(M_trn, y_trn, M_dsts)]


users = train.username.reset_index(drop=True)
EXTRAS = []  # per fold: (train-part, validation-part, test) DataFrames
t0 = time.time()
for k, (trn, val) in enumerate(folds):
    ku, yt = users.iloc[trn].reset_index(drop=True), y[trn]
    parts = [(account_stats_inner(ku, yt), account_stats(ku, yt, users.iloc[val]), account_stats(ku, yt, test.username))]
    parts.append(tuple(linear_stack(Mtr[trn], yt, [Mtr[val], Mte], "txt", alpha=3.0, C=2.0)))
    for name in ("indobertweet", "e5"):
        Etr, Ete = EMB[name]
        parts.append(tuple(linear_stack(Etr[trn], yt, [Etr[val], Ete], name, alpha=300.0, C=0.05)))
    EXTRAS.append(tuple(pd.concat([p[i] for p in parts], axis=1) for i in range(3)))
    print(f"fold {k}: stacked features ready ({time.time() - t0:.0f}s)")
print("stacked columns:", list(EXTRAS[0][0].columns))

fold 0: stacked features ready (57s)
fold 1: stacked features ready (121s)
fold 2: stacked features ready (180s)
fold 3: stacked features ready (242s)
fold 4: stacked features ready (299s)
stacked columns: ['te_user_n', 'te_user_mean_s', 'te_user_zero_s', 'te_user_mean', 'te_user_std', 'te_user_max', 'te_user_median', 'te_user_nzmean', 'txt_ridge', 'txt_pzero', 'indobertweet_ridge', 'indobertweet_pzero', 'e5_ridge', 'e5_pzero']


## 6. Models

All models predict `log1p(likes)`. Each one trains on the 5 folds with early stopping on the fold's validation part. The out-of-fold predictions are kept for blending, and the test predictions are averaged over the folds.

| Model | Stacked inputs (in addition to the unsupervised features) |
|---|---|
| `lgb_text` | account stats + TF-IDF stack |
| `lgb_text_e5` | + E5 stack |
| `lgb_text_ibt_e5` | + IndoBERTweet and E5 stacks |
| `lgb_hurdle` | account stats + TF-IDF + IndoBERTweet stacks; P(likes > 0) × E[log1p(likes) given likes > 0] |
| `xgb` | same inputs as `lgb_hurdle` |
| `cat` | same inputs as `lgb_hurdle` |
| `cat_username` | same as `cat`, plus the username as a CatBoost categorical feature (ordered target statistics) |

In [8]:
LGB_PARAMS = dict(objective="regression", learning_rate=0.03, num_leaves=31, min_child_samples=20,
                  feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  verbose=-1, n_jobs=-1, seed=SEED)
STOP = [lgb.early_stopping(200, verbose=False)]


def fit_lgb(A, ya, B, yb, C):
    m = lgb.train(LGB_PARAMS, lgb.Dataset(A, ya), 5000, valid_sets=[lgb.Dataset(B, yb)], callbacks=STOP)
    return m.predict(B, num_iteration=m.best_iteration), m.predict(C, num_iteration=m.best_iteration)


def fit_hurdle(A, ya, B, yb, C):
    clf = lgb.train({**LGB_PARAMS, "objective": "binary"}, lgb.Dataset(A, (ya > 0).astype(int)), 5000,
                    valid_sets=[lgb.Dataset(B, (yb > 0).astype(int))], callbacks=STOP)
    na, nb = ya > 0, yb > 0
    reg = lgb.train(LGB_PARAMS, lgb.Dataset(A[na], ya[na]), 5000, valid_sets=[lgb.Dataset(B[nb], yb[nb])], callbacks=STOP)
    pred = lambda D: clf.predict(D, num_iteration=clf.best_iteration) * reg.predict(D, num_iteration=reg.best_iteration)
    return pred(B), pred(C)


def fit_xgb(A, ya, B, yb, C):
    m = xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist", learning_rate=0.03, max_depth=6,
                         min_child_weight=5, subsample=0.8, colsample_bytree=0.7, reg_lambda=1.0, n_estimators=5000,
                         early_stopping_rounds=200, enable_categorical=True, random_state=SEED, n_jobs=-1)
    m.fit(A, ya, eval_set=[(B, yb)], verbose=False)
    return m.predict(B), m.predict(C)


def make_fit_cat(cat_cols):
    from catboost import CatBoostRegressor, Pool

    def fit(A, ya, B, yb, C):
        A, B, C = [D.astype({c: str for c in cat_cols}) for D in (A, B, C)]
        m = CatBoostRegressor(loss_function="RMSE", learning_rate=0.05, depth=6, iterations=5000, od_type="Iter",
                              od_wait=200, random_seed=SEED, verbose=0, thread_count=-1, allow_writing_files=False)
        m.fit(Pool(A, ya, cat_features=cat_cols), eval_set=Pool(B, yb, cat_features=cat_cols), use_best_model=True)
        return m.predict(B), m.predict(C)
    return fit


def run_cv(name, fit, prefixes, with_username=False):
    t0 = time.time()
    oof, pt = np.zeros(len(y)), np.zeros(len(test))
    for k, (trn, val) in enumerate(folds):
        ea, eb, ec = EXTRAS[k]
        keep = [c for c in ea.columns if c.startswith(prefixes)]
        A = Xtr.iloc[trn].reset_index(drop=True).join(ea[keep])
        B = Xtr.iloc[val].reset_index(drop=True).join(eb[keep])
        C = Xte.join(ec[keep])
        if with_username:
            A["username"], B["username"], C["username"] = users.iloc[trn].values, users.iloc[val].values, test.username.values
        pb, pc = fit(A, y[trn], B, y[val], C)
        oof[val], pt = np.clip(pb, 0, None), pt + np.clip(pc, 0, None) / N_FOLDS
    scores = [rmse(y[v], oof[v]) for _, v in folds]
    print(f"{name:16s} CV RMSLE {np.mean(scores):.5f} ± {np.std(scores):.5f}  ({time.time() - t0:.0f}s)")
    return oof, pt


ACC, TXT, IBT, E5 = ("te_user",), ("te_user", "txt"), ("te_user", "txt", "indobertweet"), ("te_user", "txt", "e5")
CAT_COLS = ["media_type", "client_device", "date_fmt", "acct_type"]
MODELS = {
    "lgb_text": (fit_lgb, TXT, False),
    "lgb_text_e5": (fit_lgb, E5, False),
    "lgb_text_ibt_e5": (fit_lgb, ("te_user", "txt", "indobertweet", "e5"), False),
    "lgb_hurdle": (fit_hurdle, IBT, False),
    "xgb": (fit_xgb, IBT, False),
    "cat": (make_fit_cat(CAT_COLS), IBT, False),
    "cat_username": (make_fit_cat(CAT_COLS + ["username"]), IBT, True),
}
OOF, PRED = {}, {}
for name, (fit, prefixes, with_user) in MODELS.items():
    OOF[name], PRED[name] = run_cv(name, fit, prefixes, with_user)

lgb_text         CV RMSLE 1.34082 ± 0.02474  (22s)
lgb_text_e5      CV RMSLE 1.33936 ± 0.02816  (18s)
lgb_text_ibt_e5  CV RMSLE 1.33809 ± 0.02504  (14s)
lgb_hurdle       CV RMSLE 1.33529 ± 0.02567  (24s)
xgb              CV RMSLE 1.33931 ± 0.02585  (33s)
cat              CV RMSLE 1.34034 ± 0.02262  (506s)
cat_username     CV RMSLE 1.34200 ± 0.02370  (316s)


## 7. Blend and submission

The blend weights are non-negative, sum to 1, and minimise out-of-fold RMSE in log space. The blend score itself is checked with a **second-level CV**: the weights are fit on the out-of-fold predictions of 4 folds and scored on the 5th. The final weights are then fit on all out-of-fold predictions and applied to the fold-averaged test predictions. Predictions are clipped to `[0, max(train log1p likes)]` and converted back to the like scale with `expm1`.

In [9]:
names = list(MODELS)
P = np.column_stack([OOF[n] for n in names])
T = np.column_stack([PRED[n] for n in names])


def fit_weights(P_, y_):
    k = P_.shape[1]
    res = minimize(lambda w: np.mean((P_ @ w - y_) ** 2), np.full(k, 1 / k), method="SLSQP",
                   bounds=[(0, 1)] * k, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}])
    return res.x


blend_scores = []
for trn, val in folds:
    w_k = fit_weights(P[trn], y[trn])
    blend_scores.append(rmse(y[val], np.clip(P[val] @ w_k, 0, None)))
weights = fit_weights(P, y)

summary = pd.DataFrame({"CV RMSLE (OOF)": [rmse(y, OOF[n]) for n in names], "blend weight": weights.round(3)}, index=names)
print(summary.to_string())
print(f"\nBLEND second-level CV RMSLE: {np.mean(blend_scores):.5f} ± {np.std(blend_scores):.5f}  folds {np.round(blend_scores, 5)}")

                 CV RMSLE (OOF)  blend weight
lgb_text               1.341043         0.090
lgb_text_e5            1.339655         0.182
lgb_text_ibt_e5        1.338322         0.072
lgb_hurdle             1.335533         0.389
xgb                    1.339562         0.000
cat                    1.340528         0.246
cat_username           1.342211         0.020

BLEND second-level CV RMSLE: 1.32926 ± 0.02466  folds [1.29212 1.31093 1.36025 1.33577 1.34723]


In [10]:
pred_log = np.clip(T @ weights, 0, y.max())

# sanity checks
assert len(pred_log) == len(sample) == len(test) and np.isfinite(pred_log).all()
assert (test["id"].to_numpy() == sample["id"].to_numpy()).all(), "test ids are not in sample_submission order"
oof_blend = np.clip(P @ weights, 0, None)
print(f"test predictions (log1p): mean {pred_log.mean():.3f}, std {pred_log.std():.3f}")
print(f"OOF blend      (log1p): mean {oof_blend.mean():.3f}, std {oof_blend.std():.3f}")
print(f"train target   (log1p): mean {y.mean():.3f}, std {y.std():.3f}")

submission = sample.copy()
submission[sample.columns[1]] = np.expm1(pred_log)
submission.to_csv(OUT / "submission.csv", index=False)
print("\nwrote", OUT / "submission.csv", submission.shape, list(submission.columns))
print(submission.head())
print(f"\ntotal runtime: {(time.time() - T0) / 60:.1f} min")

test predictions (log1p): mean 3.858, std 1.881
OOF blend      (log1p): mean 3.923, std 1.905
train target   (log1p): mean 3.917, std 2.336

wrote c:\Users\nayaj\Downloads\adikara-2026-tweet-likes\submission\submission.csv (5969, 2) ['id', 'likes']
      id       likes
0  26189   90.530313
1  26190  105.143148
2  26191   89.903702
3  26192  161.921952
4  26193    7.072756

total runtime: 95.9 min
